In [2]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv("../data/cleaned/cleaned.csv")

In [4]:
ORIGINAL_ROWS = len(df)
print(f"Original shape: {df.shape}")

print(f"Original columns: {df.columns}")

Original shape: (149939, 19)
Original columns: Index(['Campaign_Type', 'Target_Audience', 'Duration', 'Channel_Used',
       'Impressions', 'Clicks', 'Leads', 'Conversions', 'Revenue',
       'Acquisition_Cost', 'ROI', 'Engagement_Score', 'Customer_Segment',
       'Brand', 'CTR', 'Lead_Rate', 'Conversion_Rate', 'Rev_per_Conv',
       'Total_Cost'],
      dtype='str')


In [5]:
cols_to_drop = [
    "Brand",
    "CTR",
    "Lead_Rate",
    "Conversion_Rate",
    "Rev_per_Conv",
]
df.drop(columns=cols_to_drop, inplace=True)
print(f"After column drop: {df.columns.tolist()}")


After column drop: ['Campaign_Type', 'Target_Audience', 'Duration', 'Channel_Used', 'Impressions', 'Clicks', 'Leads', 'Conversions', 'Revenue', 'Acquisition_Cost', 'ROI', 'Engagement_Score', 'Customer_Segment', 'Total_Cost']


In [6]:
def get_bounds(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    return Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
 
def winsorize_iqr(series):
    lo, hi = get_bounds(series)
    return series.clip(lo, hi).round()
 
for col in ["Acquisition_Cost", "Revenue", "Conversions", "Leads"]:
    df[col] = winsorize_iqr(df[col])
    print(f"  '{col}' capped → min: {df[col].min():,.2f}, max: {df[col].max():,.2f}")



  'Acquisition_Cost' capped → min: 8.00, max: 911.00
  'Revenue' capped → min: 3,895.00, max: 1,448,850.00
  'Conversions' capped → min: 17.00, max: 2,912.00
  'Leads' capped → min: 48.00, max: 5,330.00


In [7]:
bounds = {
    col: get_bounds(df[col])
    for col in df.select_dtypes(include="number").columns
}

In [8]:
for col in ["Duration", "Clicks", "Total_Cost"]:
    lo, hi = bounds[col]
    before = len(df)
    df = df[(df[col] >= lo) & (df[col] <= hi)]
    print(f"  Removed {before - len(df):,} outlier rows from '{col}'")
 
print(f"After row removal: {len(df):,} rows remaining")
 

  Removed 0 outlier rows from 'Duration'
  Removed 886 outlier rows from 'Clicks'
  Removed 218 outlier rows from 'Total_Cost'
After row removal: 148,835 rows remaining


In [9]:
df["Total_Cost"] = df["Acquisition_Cost"] * df["Conversions"]
 
df["ROI"] = (
    (df["Revenue"] - df["Total_Cost"])
    / df["Total_Cost"]
).round(2)

In [10]:
lo_r, hi_r = bounds["ROI"]
df["ROI"] = df["ROI"].clip(lo_r, hi_r).round(2)
print(f"  'ROI' capped → min: {df['ROI'].min()}, max: {df['ROI'].max()}")

  'ROI' capped → min: -0.86, max: 8.91


In [11]:
print("\n=== OUTLIER VERIFICATION ===")
all_clean = True
for col in df.select_dtypes(include="number").columns:
    s = df[col]
    lo, hi = bounds[col]
    n_below = int((s < lo).sum())
    n_above = int((s > hi).sum())
    total = n_below + n_above
    if total == 0:
        print(f"  {col}: CLEAN  [{s.min():.2f} – {s.max():.2f}]")
    else:
        pct = total / len(s) * 100
        print(f"  {col}: {total} outliers ({pct:.2f}%)  below={n_below}  above={n_above}")
        all_clean = False




=== OUTLIER VERIFICATION ===
  Duration: CLEAN  [5.00 – 30.00]
  Impressions: CLEAN  [10001.00 – 100000.00]
  Clicks: CLEAN  [202.00 – 13501.00]
  Leads: 4135 outliers (2.78%)  below=0  above=4135
  Conversions: CLEAN  [17.00 – 2912.00]
  Revenue: CLEAN  [3895.00 – 1448850.00]
  Acquisition_Cost: 13043 outliers (8.76%)  below=0  above=13043
  ROI: CLEAN  [-0.86 – 8.91]
  Engagement_Score: CLEAN  [3.00 – 31.00]
  Total_Cost: CLEAN  [15487.00 – 424264.00]


In [12]:
print(f"\nAll columns clean : {all_clean}")
print(f"Final shape       : {df.shape}")
print(f"Final columns     : {df.columns.tolist()}")
print(f"Rows removed      : {ORIGINAL_ROWS - len(df):,} ({(ORIGINAL_ROWS - len(df)) / ORIGINAL_ROWS * 100:.2f}%)")
print(f"Data retained     : {len(df):,} rows ({len(df) / ORIGINAL_ROWS * 100:.2f}%)")
 
print("\nSample output:")
print(df.head())



All columns clean : False
Final shape       : (148835, 14)
Final columns     : ['Campaign_Type', 'Target_Audience', 'Duration', 'Channel_Used', 'Impressions', 'Clicks', 'Leads', 'Conversions', 'Revenue', 'Acquisition_Cost', 'ROI', 'Engagement_Score', 'Customer_Segment', 'Total_Cost']
Rows removed      : 1,104 (0.74%)
Data retained     : 148,835 rows (99.26%)

Sample output:
  Campaign_Type   Target_Audience  Duration         Channel_Used  Impressions  \
0      paid ads     working women         6     facebook, google      31869.0   
1      paid ads  premium shoppers         8  instagram, whatsapp      26640.0   
2    influencer             youth        22   facebook, whatsapp      36967.0   
3      paid ads             youth        15  instagram, whatsapp      73258.0   
4  social media  premium shoppers        11  whatsapp, instagram      91081.0   

   Clicks   Leads  Conversions  Revenue  Acquisition_Cost   ROI  \
0    3490  1946.0          902   493394              56.0  8.77   
1

In [13]:
LOG_COLS = [
    "Clicks",
    "Leads",
    "Conversions",
    "Revenue",
    "Acquisition_Cost",
]

In [14]:
print("\n=== LOG TRANSFORM RESULTS ===")
print(f"{'Column':<20}  {'Skew before':>12}  {'Skew after':>10}  {'Improvement':>12}")
print("-" * 62)
 
for col in LOG_COLS:
    skew_before = round(df[col].skew(), 2)
    df[f"{col}_log"] = np.log(df[col]).round(4)
    skew_after  = round(df[f"{col}_log"].skew(), 2)
    improvement = round(abs(skew_before) - abs(skew_after), 2)
    print(f"{col:<20}  {skew_before:>12}  {skew_after:>10}  {improvement:>+12}")



=== LOG TRANSFORM RESULTS ===
Column                 Skew before  Skew after   Improvement
--------------------------------------------------------------
Clicks                        0.77       -0.56         +0.21
Leads                         0.97        -0.5         +0.47
Conversions                   1.03       -0.48         +0.55
Revenue                       1.08       -0.47         +0.61
Acquisition_Cost              1.13       -0.17         +0.96


In [15]:
df["Profit_Label"] = (df["ROI"] > 0).astype(int)
# 1 = Profit, 0 = Loss


In [46]:
df.head()

,Campaign_Type,Target_Audience,Duration,Channel_Used,Impressions,Clicks,Leads,Conversions,Revenue,Acquisition_Cost,ROI,Engagement_Score,Customer_Segment,Total_Cost,Clicks_log,Leads_log,Conversions_log,Revenue_log,Acquisition_Cost_log,Profit_Label
0,paid ads,working women,6,"facebook, google",31869.0,3490,1946.0,902,493394,55.93,8.78,20,premium shoppers,50448.86,8.1577,7.5735,6.8046,13.1091,4.0241,1
1,paid ads,premium shoppers,8,"instagram, whatsapp",26640.0,2441,741.0,314,108644,197.44,0.75,13,college students,61996.16,7.8002,6.6080,5.7494,11.5958,5.2854,1
2,influencer,youth,22,"facebook, whatsapp",36967.0,4997,1025.0,653,318011,234.59,1.08,18,college students,153187.27,8.5166,6.9324,6.4816,12.6698,5.4578,1
3,paid ads,youth,15,"instagram, whatsapp",73258.0,5893,2565.0,1970,1341570,103.76,5.56,14,college students,204407.20,8.6815,7.8497,7.5858,14.1094,4.6421,1
4,social media,premium shoppers,11,"whatsapp, instagram",91081.0,6393,3282.0,2582,787510,78.61,2.88,13,college students,202971.02,8.7630,8.0962,7.8563,13.5766,4.3645,1


In [16]:
from sklearn.preprocessing import MultiLabelBinarizer, LabelEncoder
mlb = MultiLabelBinarizer()

df = df.reset_index(drop=True)


In [17]:
channel_lists = df['Channel_Used'].apply(lambda x: [c.strip() for c in x.split(',')])
channel_lists

0                  [facebook, google]
1               [instagram, whatsapp]
2                [facebook, whatsapp]
3               [instagram, whatsapp]
4               [whatsapp, instagram]
                     ...             
148830    [whatsapp, youtube, google]
148831                     [facebook]
148832            [google, instagram]
148833          [facebook, instagram]
148834                    [instagram]
Name: Channel_Used, Length: 148835, dtype: object

In [18]:
# Fit and transform — gives one binary column per unique channel
channel_encoded = pd.DataFrame(
    mlb.fit_transform(channel_lists),
    columns=[f"channel_{c}" for c in mlb.classes_]
)

In [19]:
df = pd.concat([df.drop(columns='Channel_Used'), channel_encoded], axis=1)

In [20]:
# ── Label Encoding: Single-value categoricals ────────────────
for col in ['Campaign_Type', 'Target_Audience', 'Customer_Segment']:
    le = LabelEncoder()
    df[f"{col}_encoded"] = le.fit_transform(df[col].str.strip().str.lower())
    df.drop(columns=col, inplace=True)

In [21]:
print(df.shape)
print(df.head(3))

df.to_csv("../data/processed/df_model_ready.csv", index=False)

(148835, 25)
   Duration  Impressions  Clicks   Leads  Conversions  Revenue  \
0         6      31869.0    3490  1946.0          902   493394   
1         8      26640.0    2441   741.0          314   108644   
2        22      36967.0    4997  1025.0          653   318011   

   Acquisition_Cost   ROI  Engagement_Score  Total_Cost  ...  Profit_Label  \
0              56.0  8.77                20     50512.0  ...             1   
1             197.0  0.76                13     61858.0  ...             1   
2             235.0  1.07                18    153455.0  ...             1   

   channel_email  channel_facebook  channel_google  channel_instagram  \
0              0                 1               1                  0   
1              0                 0               0                  1   
2              0                 1               0                  0   

   channel_whatsapp  channel_youtube  Campaign_Type_encoded  \
0                 0                0                 

In [23]:
import pickle
import json
import os

# Create parameters folder
os.makedirs("../model/parameters", exist_ok=True)

# Save label mappings
le_maps = {
    "Campaign_Type": {
        "email": 0, "influencer": 1, "paid ads": 2, "seo": 3, "social media": 4
    },
    "Target_Audience": {
        "college students": 0, "premium shoppers": 1,
        "tier 2 city customers": 2, "working women": 3, "youth": 4
    },
    "Customer_Segment": {
        "college students": 0, "premium shoppers": 1,
        "tier 2 city customers": 2, "working women": 3, "youth": 4
    }
}

with open("../model/parameters/label_mappings.json", "w") as f:
    json.dump(le_maps, f, indent=2)

print("✅ label_mappings.json saved")


✅ label_mappings.json saved
